In [ ]:
%env ALABOS_CONFIG_PATH=../src/alab_gpss/system/alabos_config_example.toml
%env SIM_MODE_FLAG=True

In [ ]:
import logging
from monty.serialization import loadfn
logging.basicConfig(level=logging.WARNING)

exp_plans = []


In [ ]:
from pymatgen.core import Composition
from pprint import pprint
import json


all_melting_points = {
    "Fe2+": 677,
    "Sn2+": 247,
    "Co2+": 726,
    "Zn2+": 290,
    "Zr4+": 437,
    "Cr2+": 824, 
    "Nb5+": 204,
    "Ca2+": 772,
    "Cr3+": 1152,
    "Mg2+": 714,
    "Hf4+": 432,
    "Ta5+": 220,
    "In3+": 586,
    "Y3+": 721,
    "Sc3+": 960,
    "Mo5+": 194,
    "Bi3+": 227,
    "Mn2+": 654,
    "Ni2+": 1001,
    "V3+": 350,
    "Li1+": 605,
    "Cu1+": 423,
    "Cu2+": 620,
    "Ho3+": 718,
    "Er3+": 774,
}

available_precursors = {
    "Li": [1],
    "Cl": [-1],
    "F": [-1],
    "Br": [-1],
    "I": [-1],
    "Fe": [2],
    "Co": [2],
    "Ni": [2],
    "Cu": [1, 2],
    "Zn": [2],
    "Sn": [2],
    "Zr": [4],
    "Hf": [4],
    "V": [3],
    "Cr": [2, 3],
    "Mg": [2],
    "Nb": [5],
    "Ca": [2],
    "Ta": [5],
    "In": [3],
    "Y": [3],
    "Sc": [3],
    "Mo": [5],
    "Bi": [3],
    "Mn": [2],
    "Ho": [3],
    "Er": [3],
}

assert sum(len(v) for v in available_precursors.values()) - 4 == len(all_melting_points)

def estimate_heating_temperature(composition):
    composition = Composition(Composition(composition).get_integer_formula_and_factor()[0])
    composition = composition.add_charges_from_oxi_state_guesses(available_precursors)
    all_metal = {e: v for e, v in composition.items() if e.is_metal}
    melting_points = {e: all_melting_points[f'{e.symbol}{int(e.oxi_state)}+'] for e in all_metal.keys()}

    weighted_average = (sum([v * (melting_points[k] + 273.15) for k, v in all_metal.items()]) / sum(all_metal.values()))
    coefficient = 3/4
    temp = weighted_average * coefficient - 273.15
    # Round to nearest 50
    rounded_temp = int(round(temp / 100.0) * 100)
    return rounded_temp


def get_precursors(composition):
    composition = Composition(Composition(composition).get_integer_formula_and_factor()[0])
    composition = composition.add_charges_from_oxi_state_guesses(available_precursors)
    all_metal = {e: v for e, v in composition.items() if e.is_metal}

    unavailable_metals = []
    for metal in all_metal.keys():
        if metal.symbol not in available_precursors:
            unavailable_metals.append(metal.symbol)

    if unavailable_metals:
        return "unavailable metals: " + ", ".join(unavailable_metals)
    
    precursor_dict = []
    multi_valence_triggered = False
    for metal, amt in all_metal.items():
        if metal.oxi_state == 0:
            unavailable_metals.append(metal.symbol)
        precursor_dict.append(Composition(f"{metal.symbol}Cl{int(metal.oxi_state)}"))
        if metal.symbol != "Li" and not multi_valence_triggered:
            multi_valence_triggered = True
            precursor_dict.append(Composition(f"{metal.symbol}Cl{int(metal.oxi_state)+1}"))

    if unavailable_metals:
        return "unavailable valence of metals: " + ", ".join(unavailable_metals)
    return precursor_dict


def prepare_previous_experiment(previous_experiment_path: str, drop_reflection: bool = False) -> str:
    previous_experiment = loadfn(previous_experiment_path)
    if drop_reflection:
        updated_previous_experiment = []
        for exp in previous_experiment:
            exp.pop("reflection", None)
            updated_previous_experiment.append(exp)
        previous_experiment = updated_previous_experiment
    return json.dumps(previous_experiment)


def pretty_formula(comp):
    comp = Composition(comp)
    comp = comp.remove_charges()
    el_amt = dict(comp.get_el_amt_dict())

    # Separate Li, Cl, metals
    li_amt = el_amt.pop("Li", 0)
    cl_amt = el_amt.pop("Cl", 0)
    metals = {e: v for e, v in el_amt.items() if e != "Li" and e != "Cl"}  # Could be all remaining
    
    # Sort metals alphabetically
    metals_sorted = sorted(metals.items())

    # Build formula string: Li, then each metal, then Cl
    pieces = []
    if li_amt:
        pieces.append(f"Li{li_amt:g}" if li_amt != 1 else "Li")
    for el, amt in metals_sorted:
        pieces.append(f"{el}{amt:g}" if amt != 1 else el)
    if cl_amt:
        pieces.append(f"Cl{cl_amt:g}" if cl_amt != 1 else "Cl")
    return ''.join(pieces)


def generate_comp_dict(composition, target_heating_temperature: int | None = None, metadata: dict | None = None, tags: str | None = None):
    """
    Given a composition string, return a dict of the form:
    {
        'pretty_formula': {
            'heating_profile': [[predicted_heating_temperature, 5, 12*60]],
            'precursors': [pretty_formula(p) for p in precursors]  # calculated from get_precursors
        }
    }
    heating temperature is predicted using estimate_heating_temperature(composition)
    precursors are computed using get_precursors(composition)
    """
    pf = pretty_formula(composition)
    precursors = get_precursors(composition)
    if target_heating_temperature is not None:
        predicted_temp = target_heating_temperature
    else:
        predicted_temp = estimate_heating_temperature(composition)

    if isinstance(precursors, str):
        raise Exception(f"Failed due to {precursors}")
    precursors_list = [pretty_formula(p) for p in precursors]
    return {
            "composition": composition,
            "heating_profile": [[predicted_temp, 2, 12*60]],
            "precursors": precursors_list,
            "metadata": metadata or {},
            "tags": tags or [],
        }

In [ ]:
from pathlib import Path


def get_latest_file(file_name_pattern: str):
    files = Path(".").glob(file_name_pattern)
    return max(files, key=lambda x: x.stat().st_mtime)

## Abnormality

In [ ]:
from pathlib import Path
from monty.serialization import loadfn
from pymatgen.core import Composition
import warnings

previous_experiment = loadfn("../data/dataset.json")
result_file = get_latest_file("experiment_design_result_*.json")
print(result_file)

all_explored_compositions = set()
for exp in previous_experiment:
    all_explored_compositions.add((Composition(exp["composition"]).reduced_composition, exp["synthesis_temperature"]))

result = loadfn(result_file)

for exp in sum(result[1]["experiments"][1:], []):
    target_comp = exp["target_composition"]
    try:
        target_composition = Composition(target_comp)
    except Exception as e:
        print(f"Error parsing {target_comp}: {e}")
        continue

    justifications = exp["justification"]
    max_heating_temperature = round(exp["max_heating_temperature"] / 50) * 50

    print(f"target_comp: {target_comp}, max_heating_temperature: {max_heating_temperature}")
    try:
        estimated_heating_temperature = estimate_heating_temperature(target_comp)
    except Exception as e:
        warnings.warn(f"Error estimating heating temperature for {target_comp}: {e}")
        continue

    if max_heating_temperature > estimated_heating_temperature + 200:
        warnings.warn(f"Skipping {target_comp} because max_heating_temperature is too high")
        continue

    if (Composition(target_comp).reduced_composition, max_heating_temperature) in all_explored_compositions:
        warnings.warn(f"Skipping {target_comp} because it has been explored")
        continue

    exp_plan = generate_comp_dict(target_comp, max_heating_temperature, metadata={"justification": justifications, "previous_experiment": Path("../data/dataset.json").read_text(), "result_file": result_file.read_text(), "result_file_path": str(result_file)}, tags=["gpt-5-auto", "abnormality"])
    exp_plans.append(exp_plan)

## Exploration

In [ ]:
from pathlib import Path
from monty.serialization import loadfn
from pymatgen.core import Composition
import warnings

previous_experiment = loadfn("../data/dataset.json")
result_file = get_latest_file("new_material_proposal_result_*.json")
print(result_file)

all_explored_compositions = set()
for exp in previous_experiment:
    all_explored_compositions.add(Composition(exp["composition"]).reduced_composition)

result = loadfn(result_file)

for exp in result["material_proposals"]:
    target_comp = exp["composition"]
    if Composition(target_comp).reduced_composition in all_explored_compositions:
        warnings.warn(f"Skipping {target_comp} because it has been explored")
        continue
    target_composition = Composition(target_comp)
    justifications = exp["justification"]
    try:
        exp_plan = generate_comp_dict(target_comp, None, metadata={"justification": justifications, "previous_experiment": Path("../data/dataset.json").read_text(), "result_file": result_file.read_text(), "result_file_path": str(result_file)}, tags=["gpt-5-auto", "exploration"])
    except Exception as e:
        print(f"Error generating {target_comp}: {e}")
        continue
    exp_plans.append(exp_plan)

## Exploration with BO

In [ ]:
from pathlib import Path
from monty.serialization import loadfn
from pymatgen.core import Composition
import warnings
from check_status import is_all_samples_finished

previous_experiment = loadfn("../data/dataset.json")
result_file = get_latest_file("bo_new_material_proposal_*.json")
print(result_file)

all_explored_compositions = set()
for exp in previous_experiment:
    all_explored_compositions.add(Composition(exp["composition"]).reduced_composition)

result = loadfn(result_file)

for exp in result["material_proposals"]:
    target_comp = exp["composition"]
    if Composition(target_comp).reduced_composition in all_explored_compositions:
        warnings.warn(f"Skipping {target_comp} because it has been explored")
        continue
    target_composition = Composition(target_comp)
    justifications = exp["justification"]
    try:
        exp_plan = generate_comp_dict(target_comp, None, metadata={"justification": justifications, "previous_experiment": Path("../data/dataset.json").read_text(), "result_file": result_file.read_text(), "result_file_path": str(result_file)}, tags=["gpt-5-auto", "exploration-bo"])
    except Exception as e:
        print(f"Error generating {target_comp}: {e}")
        continue
    exp_plans.append(exp_plan)

## Pending experiments

In [ ]:
pending_experiments = loadfn("pending_experiment.json")

for exp in pending_experiments:
    print(f"Load pending experiments: {exp}")
    exp_plans.append(exp)

In [ ]:
# Remove duplicate experiments in exp_plans (deduplicate by 'composition' and, if present, 'temperature')

unique_experiments = {}
for exp in exp_plans:
    key = (exp.get("composition"), exp.get("heating_profile")[0][0])
    if key not in unique_experiments:
        unique_experiments[key] = exp
    else:
        print(f"Duplicate experiment: {exp['composition']}")

exp_plans = list(unique_experiments.values())


In [ ]:
import time
import requests
import warnings

from monty.serialization import dumpfn

from alab_management.builders import ExperimentBuilder
from alab_gpss.system.tasks.add_sample import GPSSAddSample
from alab_gpss.system.tasks.heating import GPSSHeating
from alab_gpss.system.tasks.powder_dispensing import GPSSPowderDispensing
from alab_gpss.system.tasks.powder_mixing import GPSSPowderMixing
from alab_gpss.system.tasks.remove_sample import RemoveSample
from alab_gpss.system.tasks.sample_grinding_xrd import GPSSSampleGrindingXRD
from alab_gpss.experiment_design.reactions.balance import generate_recipe


def check_inventory(chemical: str) -> bool:
    """
    Check if the requested chemicals are in stock.
    Returns a dictionary {chemical: available(bool)}.

    Args:
        chemicals: list[str]. A list of chemicals for checking.

    Returns:
        dict: A dictionary {chemical: available(bool)}.

    Example:
        check_inventory(["NaCl", "KCl"])
        # Returns: {"NaCl": True, "KCl": True}
    """
    import urllib3
    from requests.packages.urllib3.exceptions import InsecureRequestWarning

    # Suppress only the single InsecureRequestWarning from urllib3 needed here.
    urllib3.disable_warnings(InsecureRequestWarning)

    all_chemicals_response = requests.get("https://aragorn:8000/api/dosing-head", verify=False).json()
    all_chemicals = set()
    for chemical_info in all_chemicals_response:
        if chemical_info["status"] in ["normal", "in_use"]:
            all_chemicals.add(chemical_info["chemical"])
    return chemical in all_chemicals


def to_tuple(obj):
    """Convert a nested list to a tuple"""
    if isinstance(obj, list):
        return tuple(to_tuple(item) for item in obj)
    return obj


today = time.strftime("%m%d%y")

exps = []
# exp = ExperimentBuilder(name=f"HiSpin-{today}", tags=["HiSpin"])

heating_profile_groups = {}
# group by heating profile
for i, target_info in enumerate(exp_plans):
    target = target_info["composition"]
    recipe = generate_recipe(target, target_info["precursors"], target_mass_g=0.5)
    has_missing_precursors = False
    for p in recipe.precursors:
        if p.name == "ScCl3" and p.mass >= 0.2:
            warnings.warn(f"ScCl3 mass is too high: {p.mass}. Skipping this experiment.")
            has_missing_precursors = True
            break
        if not check_inventory(p.name):
            warnings.warn(f"Chemical {p.name} is not available!")
            has_missing_precursors = True
    if has_missing_precursors:
        continue
    heating_profile = to_tuple(target_info["heating_profile"])
    if heating_profile not in heating_profile_groups:
        heating_profile_groups[heating_profile] = []
    heating_profile_groups[heating_profile].append(i)

pending_experiments = []
print(f"Number of heating batches: {len(heating_profile_groups)}")
for idx, (heating_profile, targets) in enumerate(heating_profile_groups.items(), 1):
    if 0 < len(targets) % 6 <= 2:
        number_sample_to_pending = len(targets) % 6
        for target in targets[-number_sample_to_pending:]:
            pending_experiments.append(exp_plans[target])
        heating_profile_groups[heating_profile] = heating_profile_groups[heating_profile][:-number_sample_to_pending]
        print(f"  Batch {idx}: {heating_profile} ({len(targets) - len(targets) % 6} sample(s)) - Put {len(targets) % 6} samples to pending experiments due to small number of samples (<=2)")
    else:
        print(f"  Batch {idx}: {heating_profile} ({len(targets)} sample(s))")

for heating_profile, targets_ in heating_profile_groups.items():
    for i in range(0, len(targets_), 6):
        exp = ExperimentBuilder(name=f"HiSpin-{today}", tags=["HiSpin"])
        targets = targets_[i:i+6]
        samples = []
        for target in targets:
            target_info = exp_plans[target]
            target = exp_plans[target]["composition"]
            tags = target_info["tags"]
            sample = exp.add_sample(name=f"{target.replace('.', 'p')}_{today}", tags=["HiSpin", *tags], **target_info["metadata"])
            recipe = generate_recipe(target, target_info["precursors"], target_mass_g=0.5)

            for p in recipe.precursors:
                if p.mass <= 0:
                    raise ValueError(f"The mass of {p.name} is non-positive!")

            samples.append(sample)
            print(recipe)
            add_sample = GPSSAddSample(notify_user=False)
            add_sample.add_to(sample)
            powder_dispensing = GPSSPowderDispensing({p.name: p.mass for p in recipe.precursors}, 1, num_balls=4)
            powder_dispensing.add_to(sample)
            powder_mixing = GPSSPowderMixing([1000, 1500], [300, 300], interval_seconds=60)
            powder_mixing.add_to(sample)

        heating = GPSSHeating(heating_profile)
        heating.add_to(samples)

        for sample in samples:
            sample_grinding_xrd = GPSSSampleGrindingXRD(240, 28)
            sample_grinding_xrd.add_to(sample)
            remove_sample = RemoveSample()
            remove_sample.add_to(sample)
        exps.append(exp)


In [ ]:
exit()

In [ ]:
import time
for exp in exps:
    print(f"Submit experiment: {exp.name}")
    exp.submit(address="http://aragorn:8895")
    time.sleep(1)
dumpfn(pending_experiments, "pending_experiment.json")